# IslamicEval 2026 — Task 2 · Verification (GPU hybrid: neural Ayah/matn + rules isnad/claimed_source)

Upgrades the 0.845 rule system: fine-tunes **AraBERTv2** as a pair classifier
`(span [SEP] retrieved_source) → correct / incorrect` for **Ayah & matn** (the paper's RAG-verifier
idea), while keeping the strong rule verifiers for **isnad** (grounded) and **claimed_source**
(parent-linked). Metric: macro accuracy over the 4 types (official scorer).

**Resume/caching/weights:** Drive checkpoints + cache; weights → private HF repo; resumes on re-run.
GPU runtime + `HF_TOKEN` in Colab Secrets.

In [ ]:
# ---- deps ----
!pip -q install "transformers>=4.44" "datasets>=2.20" accelerate seqeval huggingface_hub rapidfuzz scikit-learn
import torch, os, json, subprocess, sys
from pathlib import Path
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - set Runtime>GPU (T4)")

In [ ]:
# ---- HF auth (use Colab Secrets: key icon -> add HF_TOKEN) + Drive for resume ----
from huggingface_hub import login, HfApi
HF_USER = "FatimahEmadEldin"                 # <-- your HF username
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")       # add token under Colab 'Secrets' (🔑) named HF_TOKEN
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
assert HF_TOKEN, "Add your HF token to Colab Secrets as HF_TOKEN (do NOT hardcode it)."
login(HF_TOKEN)
try:
    from google.colab import drive; drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive/IslamicEval2026")     # checkpoints survive disconnects here
except Exception:
    WORK = Path("/content/IslamicEval2026_work")
WORK.mkdir(parents=True, exist_ok=True); print("work dir:", WORK)

In [ ]:
# ---- clone the task repo (data + scorer) ----
REPO=Path("/content/IslamicEval2026")
if not REPO.exists():
    subprocess.run(["git","clone","--depth","1","https://github.com/Watheq9/IslamicEval2026.git",str(REPO)],check=True)
print("repo:", REPO.exists())

In [ ]:
# ---- resume + cache helpers ----
from transformers.trainer_utils import get_last_checkpoint
def last_ckpt(d):
    d=str(d)
    return get_last_checkpoint(d) if os.path.isdir(d) and any(os.scandir(d)) else None

## Config + Arabic normalization (codepoint-built) + retriever

In [ ]:
import re, numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
from rapidfuzz import fuzz
MODEL_NAME="aubmindlab/bert-base-arabertv2"
HF_MODEL_ID=f"{HF_USER}/islamiceval2026-task2-verifier"
TASK=Path(str(WORK))/"task2"; TASK.mkdir(parents=True,exist_ok=True)
MAX_LEN=192; EPOCHS=3; LR=2e-5; BATCH=16; SEED=42
TRAIN=REPO/"train_set/train.jsonl"; DEV=REPO/"dev_set/dev.jsonl"
GOLD=REPO/"dev_set/dev_task_2.tsv"; SCORER=REPO/"Scoring_scripts/task2_scoring.py"
_T=[(0x610,0x61A),(0x64B,0x65F),(0x670,0x670),(0x6D6,0x6DC),(0x6DF,0x6E8),(0x6EA,0x6ED)]
_TASHKEEL=re.compile('['+''.join(chr(a)+'-'+chr(b) for a,b in _T)+']'); _NA=re.compile('[^'+chr(0x621)+'-'+chr(0x64A)+'\\s]'); _SP=re.compile(r'\s+')
def norm(t):
    if not t: return ""
    t=_SP.sub(' ',_TASHKEEL.sub('',str(t)).replace(chr(0x640),'')).strip()
    t=re.sub('['+''.join(chr(c) for c in (0x622,0x623,0x625,0x627,0x671,0x621))+']',chr(0x627),t)
    t=t.replace(chr(0x649),chr(0x64A)).replace(chr(0x624),chr(0x648)).replace(chr(0x626),chr(0x64A)).replace(chr(0x629),chr(0x647))
    return _SP.sub(' ',_NA.sub(' ',t)).strip()
def rj(p):
    import json as j; return j.load(open(p,encoding="utf-8"))
def load_quran():
    o=[]
    for d in rj(REPO/"Corpora/quranic_verses.json"):
        t=d.get("ayah_text")
        if t: o.append({"text":str(t),"norm":norm(t),"surah_id":d.get("surah_id"),"surah_name":d.get("surah_name"),"ayah_id":d.get("ayah_id")})
    return o
def load_hadith():
    o=[]
    for d in rj(REPO/"Corpora/six_hadith_books.json"):
        m=d.get("Matn")
        if m:
            full=d.get("hadithTxt") or ""; nm=norm(m); nf=norm(full)
            o.append({"text":str(m),"norm":nm,"book":d.get("title"),"full_norm":nf,"chain_norm":(nf.replace(nm," ").strip() if nm and nm in nf else nf)})
    return o
QURAN=load_quran(); HADITH=load_hadith(); print("quran",len(QURAN),"hadith",len(HADITH))
class Ret:
    def __init__(s,recs):
        s.recs=recs; s.vec=TfidfVectorizer(analyzer="char_wb",ngram_range=(3,5),min_df=1); s.mat=s.vec.fit_transform([r["norm"] for r in recs])
    def best(s,spans,k=15,chunk=256,topn=1):
        qn=[norm(x) for x in spans]; res=[(0.0,None,[]) for _ in spans]; idx=[i for i,q in enumerate(qn) if q]
        if not idx: return res
        Q=s.vec.transform([qn[i] for i in idx])
        for st in range(0,len(idx),chunk):
            sub=idx[st:st+chunk]; sims=linear_kernel(Q[st:st+chunk],s.mat)
            for row,i in enumerate(sub):
                kk=min(k,sims.shape[1]); top=np.argpartition(sims[row],-kk)[-kk:]; q=qn[i]; sc=[]
                for jj in top:
                    v=max(fuzz.token_set_ratio(q,s.recs[jj]["norm"]),fuzz.partial_ratio(q,s.recs[jj]["norm"]))/100.0; sc.append((v,s.recs[jj]))
                sc.sort(key=lambda x:-x[0]); res[i]=(sc[0][0],sc[0][1],[r for _,r in sc[:topn]])
        return res
QRET=Ret(QURAN); HRET=Ret(HADITH); print("retrievers ready")

## Build (span [SEP] retrieved_source) training pairs for Ayah & matn

In [ ]:
def segments(path):
    out=[]
    for l in open(path,encoding="utf-8"):
        if not l.strip(): continue
        r=json.loads(l); ans=r.get("generated_answer") or ""
        for ann in r.get("annotations") or []:
            for s in ann.get("segments") or []:
                t=s.get("type"); a=s.get("span_start"); b=s.get("span_end")
                txt=ans[a:b] if (a is not None and b is not None and b>a) else (s.get("span_text") or "")
                out.append({"rid":r.get("id"),"aid":ann.get("annotation_id"),"type":t,"txt":txt,"gold":s.get("label")})
    return out
def pairs(segs, split):
    A=[s for s in segs if s["type"]=="Ayah" and s["gold"] in ("correct","incorrect")]
    M=[s for s in segs if s["type"]=="matn" and s["gold"] in ("correct","incorrect")]
    ex={"text_a":[],"text_b":[],"label":[]}
    for pool,ret in [(A,QRET),(M,HRET)]:
        res=ret.best([s["txt"] for s in pool])
        for s,(sc,rec,_) in zip(pool,res):
            ex["text_a"].append(norm(s["txt"])); ex["text_b"].append(rec["norm"] if rec else "")
            ex["label"].append(1 if s["gold"]=="correct" else 0)
    print(split,"pairs",len(ex["label"])); return ex
train_segs=segments(TRAIN); dev_segs=segments(DEV)
import datasets
from transformers import AutoTokenizer
tok=AutoTokenizer.from_pretrained(MODEL_NAME)
CACHE=TASK/"ds_cache"
if CACHE.exists(): ds=datasets.load_from_disk(str(CACHE)); print("cached")
else:
    dd=datasets.DatasetDict({"train":datasets.Dataset.from_dict(pairs(train_segs,"train")),
                             "dev":datasets.Dataset.from_dict(pairs(dev_segs,"dev"))})
    ds=dd.map(lambda b: tok(b["text_a"],b["text_b"],truncation=True,max_length=MAX_LEN),batched=True)
    ds.save_to_disk(str(CACHE)); print("tokenized+cached")

## Fine-tune the Ayah/matn verifier (resume-aware, push to HF)

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
model=AutoModelForSequenceClassification.from_pretrained(MODEL_NAME,num_labels=2)
args=TrainingArguments(output_dir=str(TASK/"ckpt"), eval_strategy="epoch", save_strategy="epoch", save_total_limit=2,
    num_train_epochs=EPOCHS, learning_rate=LR, per_device_train_batch_size=BATCH, per_device_eval_batch_size=32,
    weight_decay=0.01, warmup_ratio=0.06, fp16=torch.cuda.is_available(), logging_steps=100, seed=SEED,
    report_to="none", push_to_hub=True, hub_model_id=HF_MODEL_ID, hub_private_repo=True, hub_strategy="checkpoint")
trainer=Trainer(model=model,args=args,train_dataset=ds["train"],eval_dataset=ds["dev"],tokenizer=tok,data_collator=DataCollatorWithPadding(tok))
trainer.train(resume_from_checkpoint=last_ckpt(TASK/"ckpt"))
trainer.save_model(str(TASK/"best")); trainer.push_to_hub(); print("pushed",HF_MODEL_ID)

## Rule verifiers for isnad + claimed_source (our 0.845 system), then combine + score

In [ ]:
AR2EN=str.maketrans(''.join(chr(0x660+i) for i in range(10)),'0123456789')
def find_number(t):
    m=re.search(r'\d+',str(t).translate(AR2EN)); return int(m.group()) if m else None
SURAH={norm(v["surah_name"]):v["surah_id"] for v in QURAN if v.get("surah_name") and v.get("surah_id") is not None}
def _w(*c): return norm(''.join(chr(x) for x in c))
BOOKS=[_w(0x627,0x644,0x628,0x62E,0x627,0x631,0x64A),_w(0x645,0x633,0x644,0x645),_w(0x627,0x644,0x62A,0x631,0x645,0x630,0x64A),
 _w(0x627,0x644,0x646,0x633,0x627,0x626,0x64A),_w(0x627,0x628,0x646,0x20,0x645,0x627,0x62C,0x647),_w(0x627,0x62D,0x645,0x62F),_w(0x645,0x627,0x644,0x643)]
def verify_cs(span,pk,pr):
    c=norm(span)
    if pr is None or not c: return "correct"
    if pk=="Ayah":
        sid=next((v for n,v in SURAH.items() if n and len(n)>2 and n in c),None)
        if sid is None: return "correct"
        if str(sid)!=str(pr.get("surah_id")): return "incorrect"
        n=find_number(span)
        if n is not None and pr.get("ayah_id") is not None: return "correct" if str(n)==str(pr.get("ayah_id")) else "incorrect"
        return "correct"
    cb=next((b for b in BOOKS if b in c),None); tb=norm(str(pr.get("book") or ""))
    if cb is None or not tb: return "correct"
    return "correct" if (cb in tb or tb in cb) else "incorrect"
TAU_ISNAD=0.85
# neural predictions for Ayah/matn on dev
import torch, pandas as pd
def neural_pred(segs):
    A=[s for s in segs if s["type"]=="Ayah"]; M=[s for s in segs if s["type"]=="matn"]
    out={}
    for pool,ret in [(A,QRET),(M,HRET)]:
        if not pool: continue
        res=ret.best([s["txt"] for s in pool])
        ta=[norm(s["txt"]) for s in pool]; tb=[r["norm"] if r else "" for _,r,_ in res]
        enc=tok(ta,tb,truncation=True,max_length=MAX_LEN,padding=True,return_tensors="pt")
        with torch.no_grad():
            pr=[]
            for i in range(0,len(ta),64):
                b={k:v[i:i+64].to(model.device) for k,v in enc.items()}
                pr.append(model(**b).logits.argmax(-1).cpu())
            pred=torch.cat(pr).numpy()
        for s,p in zip(pool,pred): out[(s["rid"],s["aid"],s["type"])]="correct" if p==1 else "incorrect"
    return out
# parents for cs/isnad
def parents(segs):
    par={}
    A=[s for s in segs if s["type"]=="Ayah"]; M=[s for s in segs if s["type"]=="matn"]
    for pool,ret,kind in [(A,QRET,"Ayah"),(M,HRET,"matn")]:
        res=ret.best([s["txt"] for s in pool],topn=3)
        for s,(sc,rec,tops) in zip(pool,res): par[(s["rid"],s["aid"])]=(kind,rec,tops)
    return par
np_=neural_pred(dev_segs); par=parents(dev_segs)
rows=[]
for s in dev_segs:
    st=s["type"]; key=(s["rid"],s["aid"],st)
    if st in ("Ayah","matn"): lab=np_.get(key,"incorrect")
    elif st=="claimed_source":
        pk,pr,_=par.get((s["rid"],s["aid"]),(None,None,None)); lab=verify_cs(s["txt"],pk,pr)
    elif st=="isnad":
        pk,pr,tops=par.get((s["rid"],s["aid"]),(None,None,[])); q=norm(s["txt"]); fs=0.0
        if q and pk=="matn":
            for r in (tops or []):
                if r: fs=max(fs,max(fuzz.token_set_ratio(q,r.get("full_norm","")),fuzz.partial_ratio(q,r.get("full_norm","")))/100.0)
        lab="correct" if fs>=TAU_ISNAD else "incorrect"
    else: lab="incorrect"
    rows.append([s["rid"],s["aid"],st,lab])
sub=pd.DataFrame(rows,columns=["Response_ID","Annotation_ID","Segment_Type","Label"])
sub=sub[sub["Label"].isin(["correct","incorrect"])].drop_duplicates(subset=["Response_ID","Annotation_ID","Segment_Type"])
OUT="/content/submission_task2_dev.tsv"; sub.to_csv(OUT,sep="\t",index=False)
o=Path("/content/t2o"); o.mkdir(exist_ok=True)
r=subprocess.run([sys.executable,str(SCORER),"--pred",OUT,"--ref",str(GOLD),"--output",str(o),"-v"],capture_output=True,text=True)
print(r.stdout,r.stderr); print("SCORES:",(o/"scores.json").read_text())

In [ ]:
import zipfile
zp="/content/submission_task2_dev.zip"
with zipfile.ZipFile(zp,"w",zipfile.ZIP_DEFLATED) as zf: zf.write(OUT,"submission_task2_dev.tsv")
api=HfApi(); DS=f"{HF_USER}/IslamicEval2026-Subtask2-Submission"
try:
    api.upload_file(path_or_fileobj=zp,path_in_repo="task2/submission_task2_dev_gpu.zip",repo_id=DS,repo_type="dataset")
    api.upload_file(path_or_fileobj=OUT,path_in_repo="submission_dev_gpu.tsv",repo_id=DS,repo_type="dataset")
    print("pushed to",DS)
except Exception as e: print("skip:",e)

## Notes
- The neural Ayah/matn verifier learns from `(span, retrieved_source)` — add the **gated HF synthetic set**
  (`ArabicNLPWorld/arabic-islamic-hallucination-synthetic`, 543k) for far more Ayah/matn training pairs.
- Keep diacritics in the *verifier* input (TCE: +2–3 pts) — try `norm(...)` off for `text_a/text_b`.
- isnad/claimed_source stay rule-based (our 0.845 system); a narrator DB would lift isnad further.